In [0]:
CREATE CATALOG IF NOT EXISTS banking;

CREATE SCHEMA IF NOT EXISTS banking.bronze;

CREATE SCHEMA IF NOT EXISTS banking.silver;

CREATE SCHEMA IF NOT EXISTS banking.gold;

In [0]:

-- 1. Initial customer snapshot
CREATE TABLE IF NOT EXISTS banking.bronze.customer_raw (
    customer_id       BIGINT,
    first_name        STRING,
    last_name         STRING,
    email             STRING,
    phone             STRING,
    address_line      STRING,
    city              STRING,
    state             STRING,
    postal_code       STRING,
    customer_segment  STRING,
    risk_rating       STRING,
    kyc_status        STRING,
    created_at        TIMESTAMP
)
USING DELTA;


-- 2. Initial account snapshot
CREATE TABLE IF NOT EXISTS banking.bronze.account_raw (
    account_id        BIGINT,
    customer_id       BIGINT,
    account_type      STRING,
    currency          STRING,
    branch_code       STRING,
    account_status    STRING,
    available_balance DECIMAL(18,2),
    opened_date       DATE,
    last_activity_ts TIMESTAMP,
    updated_at        TIMESTAMP
)
USING DELTA;


-- 3. Customer CDC events
CREATE TABLE IF NOT EXISTS banking.bronze.customer_cdc_raw (
    event_id          STRING,
    source_sequence   BIGINT,
    customer_id       BIGINT,
    operation         STRING,
    event_ts          TIMESTAMP,
    first_name        STRING,
    last_name         STRING,
    email             STRING,
    phone             STRING,
    address_line      STRING,
    city              STRING,
    state             STRING,
    postal_code       STRING,
    customer_segment  STRING,
    risk_rating       STRING,
    kyc_status        STRING,
    created_at        TIMESTAMP
)
USING DELTA;


-- 4. Account CDC events
CREATE TABLE IF NOT EXISTS banking.bronze.account_cdc_raw (
    event_id          STRING,
    source_sequence   BIGINT,
    account_id        BIGINT,
    customer_id       BIGINT,
    operation         STRING,
    event_ts          TIMESTAMP,
    account_type      STRING,
    currency          STRING,
    branch_code       STRING,
    account_status    STRING,
    available_balance DECIMAL(18,2),
    opened_date       DATE,
    last_activity_ts TIMESTAMP,
    updated_at        TIMESTAMP
)
USING DELTA;


In [0]:
SHOW TABLES IN banking.bronze;

In [0]:

-- 1. Customer dimension (SCD Type 2)
CREATE TABLE IF NOT EXISTS banking.silver.dim_customer (
    customer_id          BIGINT,
    first_name           STRING,
    last_name            STRING,
    email                STRING,
    phone                STRING,
    address_line         STRING,
    city                 STRING,
    state                STRING,
    postal_code          STRING,
    customer_segment     STRING,
    risk_rating          STRING,
    kyc_status           STRING,
    created_at           TIMESTAMP,
    effective_start_ts   TIMESTAMP,
    effective_end_ts     TIMESTAMP,
    is_current           BOOLEAN
)
USING DELTA;


-- 2. Account dimension (SCD Type 1)
CREATE TABLE IF NOT EXISTS banking.silver.dim_account (
    account_id           BIGINT,
    customer_id          BIGINT,
    account_type         STRING,
    currency             STRING,
    branch_code          STRING,
    account_status       STRING,
    available_balance    DECIMAL(18,2),
    opened_date          DATE,
    last_activity_ts     TIMESTAMP,
    updated_at           TIMESTAMP
)
USING DELTA;


-- 3. Processed customer CDC
CREATE TABLE IF NOT EXISTS banking.silver.customer_cdc_processed (
    event_id             STRING,
    source_sequence      BIGINT,
    customer_id          BIGINT,
    operation            STRING,
    event_ts             TIMESTAMP,
    first_name           STRING,
    last_name            STRING,
    email                STRING,
    phone                STRING,
    address_line         STRING,
    city                 STRING,
    state                STRING,
    postal_code          STRING,
    customer_segment     STRING,
    risk_rating          STRING,
    kyc_status           STRING,
    created_at           TIMESTAMP
)
USING DELTA;


-- 4. Processed account CDC
CREATE TABLE IF NOT EXISTS banking.silver.account_cdc_processed (
    event_id             STRING,
    source_sequence      BIGINT,
    account_id           BIGINT,
    customer_id          BIGINT,
    operation            STRING,
    event_ts             TIMESTAMP,
    account_type         STRING,
    currency             STRING,
    branch_code          STRING,
    account_status       STRING,
    available_balance    DECIMAL(18,2),
    opened_date          DATE,
    last_activity_ts     TIMESTAMP,
    updated_at           TIMESTAMP
)
USING DELTA;


-- 5. Data quality quarantine
CREATE TABLE IF NOT EXISTS banking.silver.quarantine (
    source_table         STRING,
    record_key           STRING,
    rejection_reason     STRING,
    raw_record           STRING,
    ingestion_ts         TIMESTAMP
)
USING DELTA;


In [0]:
SHOW TABLES IN banking.silver;

In [0]:

-- 1. Customer 360
CREATE TABLE IF NOT EXISTS banking.gold.customer_360 (
    customer_id          BIGINT,
    full_name            STRING,
    email                STRING,
    phone                STRING,
    city                 STRING,
    state                STRING,
    customer_segment     STRING,
    risk_rating          STRING,
    kyc_status           STRING,
    total_accounts       BIGINT,
    active_accounts      BIGINT,
    total_balance        DECIMAL(18,2),
    last_updated_ts      TIMESTAMP
)
USING DELTA;


-- 2. Account 360
CREATE TABLE IF NOT EXISTS banking.gold.account_360 (
    account_id           BIGINT,
    customer_id          BIGINT,
    customer_name        STRING,
    account_type         STRING,
    currency             STRING,
    branch_code          STRING,
    account_status       STRING,
    available_balance    DECIMAL(18,2),
    opened_date          DATE,
    last_activity_ts     TIMESTAMP,
    last_updated_ts      TIMESTAMP
)
USING DELTA;


-- 3. Branch analytics
CREATE TABLE IF NOT EXISTS banking.gold.branch_analytics (
    branch_code          STRING,
    total_accounts       BIGINT,
    active_accounts      BIGINT,
    inactive_accounts    BIGINT,
    total_balance        DECIMAL(18,2),
    average_balance      DECIMAL(18,2),
    total_customers      BIGINT,
    last_updated_ts      TIMESTAMP
)
USING DELTA;


In [0]:
SHOW TABLES IN banking.gold;